## 📑 Core PM Modifications
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/BIArHg)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()


from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
import boto3
import os

#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ"S3PREFIX" 

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d')  

som_date = spark.sql(
f'''
SELECT 
     MAX(cal.CalendarDate) CalendarDate
FROM `{env_prefix}-cdh`.reference.ref_calendar AS cal
WHERE cal.IsBusinessDay = 1
AND cal.CalendarDate <= LAST_DAY(DATE_ADD(MONTH, -1, {run_date!r}))
''').collect()[0][0]

soq_date = spark.sql(
f'''
SELECT 
     MAX(cal.CalendarDate) CalendarDate
FROM `{env_prefix}-cdh`.reference.ref_calendar AS cal
WHERE cal.IsBusinessDay = 1
AND cal.CalendarDate <= date_add(QUARTER, -1, DATE {run_date!r})
''').collect()[0][0]


In [0]:
table_name = 'Core_PMModifications'

schema = (
    "BusinessDate date," 
    "ExposureKey string,"
    "Segment string,"
    "LTVFlag string,"
    "MtDBucket int,"
    "ReportedBucket int,"
    "AgreementKey string,"
    "CDI string,"
    "PartyTaxRegistrationNumber string,"
    "EBAIndicator string,"
    "EBAStatus string,"
    "ContactMtD string,"
    "AssetSegment string,"
    "SoMBucket int,"
    "SoMDenouncedStatus string,"
    "SoMBalance decimal(19,2),"
    "ModificationCategory string,"
    "ModificationProduct string,"
    "ImplementationDate date,"
    "NewAssignmentChannel string,"
    "AssignmentChannel string,"
    "AssignmentTeam string,"
    "AssignmentAgent string,"
    "CourseOfAction string,"
    "AssignmentSegmentDetailed string,"
    "SoMStatus string,"
    "MtDProductionStatus string,"
    "MtDProductionDate date,"
    "CurrentStatus string,"
    "ApplicationProduct string,"
    "FinalSavedFlag string,"
    "FinalSavedReason string,"
    "FinalSavedDate date,"
    "TargetDate date,"
    "NPEFinalizedInflow int,"
    "FinalAssignmentChannel string,"
    "FinalAssignmentTeam string,"
    "PaymentAttributionChannel string,"
    "PaymentAttributionTeam string,"
    "PaymentAttributionAmount decimal(19,2),"
    "ApplicationTarget decimal(19,2),"
    "ImplementationTarget decimal(19,2),"
    "SavedTarget decimal(19,2),"
    "InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)

##### Create Delta Tables with selected fields & Temporary Views for each delta table

##### Assignment Data Mapping

In [0]:
AssignmentDataMapping = spark.sql(
f'''
SELECT
			m.BusinessDate
   		,	m.ExposureKey							 
		,	m.AssignmentChannel								
		,	m.AssignmentTeam					
		,	m.AssignmentSegmentDetailed					   
		,	m.CourseOfAction								
		,	m.AssignmentAgent
		, 	m.Anchor										
		,	m.AgreementKey																
		,	m.CDI																		
		,	m.PartyTaxRegistrationNumber																
		,	m.AssetSegment
		,	CAST(m.SoMSourceBucket AS INT) AS SoMSourceBucket
		,	m.SoMBalanceAmountEUR
		,	m.EBAStatus			                
		,	CAST(m.RiskBucket AS INT) AS RiskBucket
		,	m.AttributionChannel 
		,	m.AttributionTeam
		,	m.AttributionCOA
		,	m.AttributionSegmentDetailed
		,	m.Event
		,	m.PaymentAmount
FROM `{env_prefix}-cdh-delivery`.repoint.Core_AssignmentDataMapping m
-- FROM `{env_prefix}-cdh-delivery`.repoint.core_assignmentdatamapping_test m
WHERE m.BusinessDate = {run_date!r}
'''
)
AssignmentDataMapping.createOrReplaceTempView("AssignmentDataMapping")

##### SavedInfo

In [0]:
SavedInfo = spark.sql(
f'''
SELECT * 
FROM (
SELECT DISTINCT
                BusinessDate
            ,	Account AS ExposureKey
			,   SavedFlag AS FinalSaved		   
			,	SavedDate AS FinalSavedDate   
		    ,   SavedReason
            -- ,   InflowTargetDateCalc AS TargetDate
            -- ,   InflowTargetDate
            ,   rank() over (partition by Account order by InflowTargetDateCalc desc) as rank
FROM `{env_prefix}-cdh`.derivative.dportfoliotoolcore
-- FROM `{env_prefix}-cdh-delivery`.repoint.dportfoliotoolcore_uat_27
WHERE BusinessDate = {run_date!r}
AND RelationType = 'Primary Owner') a 
WHERE a.rank = 1
'''
)
SavedInfo.createOrReplaceTempView("SavedInfo")

##### MaxInflows

In [0]:
maxinflows = spark.sql(
f'''
SELECT  ExposureNumber
    ,   TargetDate
FROM `{env_prefix}-cdh-delivery`.repoint.core_maxinflows
WHERE BusinessDate = {som_date!r}
'''
)
maxinflows.createOrReplaceTempView("maxinflows")

##### attributionchannel_restructure

In [0]:
attributionchannel_restructure = spark.sql(
f'''
SELECT * 
FROM (
SELECT DISTINCT
        ExposureKey
	,	AttributionChannel
	,	AttributionTeam
	,	AttributionCOA
	,	AttributionSegmentDetailed
	, 	rank() over (partition by ExposureKey order by AttributionChannel) Rank
FROM AssignmentDataMapping
WHERE Event IN ('Saved due to Restructure' , 'Saved due to Extension', 'Extension','Restructure','Active Application')
AND AttributionChannel IS NOT NULL) a
WHERE a.Rank = 1
'''
)
attributionchannel_restructure.createOrReplaceTempView("attributionchannel_restructure")

##### attributionchannel_payment

In [0]:
attributionchannel_payment = spark.sql(
f'''
SELECT * 
FROM (
SELECT DISTINCT
        ExposureKey
	,	AttributionChannel
	,	AttributionTeam
	,	AttributionCOA
	,	AttributionSegmentDetailed
	,	SUM(PaymentAmount) AS PaymentAmount
	, 	rank() over (partition by ExposureKey order by AttributionChannel) Rank
FROM AssignmentDataMapping
WHERE Event IN ('Saved due to Payment') 
AND AttributionChannel IS NOT NULL
GROUP BY ExposureKey,	AttributionChannel, AttributionTeam, AttributionCOA, AttributionSegmentDetailed) a
WHERE a.Rank = 1
'''
)
attributionchannel_payment.createOrReplaceTempView("attributionchannel_payment")

##### dexposure

In [0]:
dexposure = spark.sql(
f'''
SELECT
	  de.BusinessDate
    , de.ExposureKey
    , de.ExposureStatus
    , de.OutstandingTotalBalance
    , e.Bucket
    , de.LPS
    , de.IsActiveFlag
FROM `{env_prefix}-cdh`.derivative.dexposure de
INNER JOIN `{env_prefix}-cdh`.primary.exposure e
ON e.ExposureKey = de.ExposureKey 
AND e.BusinessDate = de.BusinessDate
WHERE de.BusinessDate in ({run_date!r}, {som_date!r})
AND e.PartyRelationshipType = 'Primary Owner' 
AND e.SourceSystem = 'qca'
AND e.PortfolioName = 'Core Retail'
'''
)
dexposure.createOrReplaceTempView("dexposure")

##### interaction

In [0]:
interaction = spark.sql(
f'''
select  ia.ExposureKey
    ,   ia.InteractionTypeKey
    ,   ia.InteractionType
    ,   ia.InteractionDate
    ,   ia.CreatedDate 
    ,   m.hierarchy
    ,   m.`3rd_level_category`
    , CASE WHEN left(m.`1st_level_category`,1) = '5'         THEN 4    
         WHEN left(m.`1st_level_category`,1) = '4'             THEN 4
         WHEN left(m.`1st_level_category`,1) = '3'             THEN 3
         WHEN left(m.`1st_level_category`,1) = '2'             THEN 2                      
         WHEN left(m.`2nd_level_category`,3) = '1.2'           THEN 1.2
         WHEN left(m.`2nd_level_category`,3) = '1.1'           THEN 1.1
    END AS Priority
    , CASE WHEN left(m.`1st_level_category`,1) = '5'             THEN 5    
         WHEN left(m.`1st_level_category`,1) = '4'             THEN 4
         WHEN left(m.`1st_level_category`,1) = '3'             THEN 3
         WHEN left(m.`1st_level_category`,1) = '2'             THEN 2                      
         WHEN left(m.`2nd_level_category`,3) = '1.2'           THEN 1.2
         WHEN left(m.`2nd_level_category`,3) = '1.1'           THEN 1.1
    END AS Priority_Aux    
from `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) ia
inner join `{env_prefix}-reportinputfiles`.repoint.core_qcabactivitiesmapping m on ia.InteractionTypeKey = m.Activity_Code
where ia.BusinessDate =  {run_date!r}
and ia.InteractionDate > last_day({som_date!r})
and ia.InteractionDate <= {run_date!r}
and m.Activity_Group like '%Modifications%'

union

select  ia_o.ExposureKey
    ,   ia_o.InteractionTypeKey
    ,   ia_o.InteractionType
    ,   ia_o.InteractionDate
    ,   ia_o.CreatedDate
    ,   m.hierarchy
    ,   m.`3rd_level_category`
    , CASE WHEN left(m.`1st_level_category`,1) = '5'         THEN 4    
         WHEN left(m.`1st_level_category`,1) = '4'             THEN 4
         WHEN left(m.`1st_level_category`,1) = '3'             THEN 3
         WHEN left(m.`1st_level_category`,1) = '2'             THEN 2                      
         WHEN left(m.`2nd_level_category`,3) = '1.2'           THEN 1.2
         WHEN left(m.`2nd_level_category`,3) = '1.1'           THEN 1.1
    END AS Priority
  , CASE WHEN left(m.`1st_level_category`,1) = '5'             THEN 5    
         WHEN left(m.`1st_level_category`,1) = '4'             THEN 4
         WHEN left(m.`1st_level_category`,1) = '3'             THEN 3
         WHEN left(m.`1st_level_category`,1) = '2'             THEN 2                      
         WHEN left(m.`2nd_level_category`,3) = '1.2'           THEN 1.2
         WHEN left(m.`2nd_level_category`,3) = '1.1'           THEN 1.1
    END AS Priority_Aux 
from `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) ia_o
inner join `{env_prefix}-reportinputfiles`.repoint.core_qcabactivitiesmapping m on ia_o.InteractionTypeKey = m.Activity_Code
where ia_o.BusinessDate <= {run_date!r}
and ia_o.InteractionDate > last_day({som_date!r})
and ia_o.InteractionDate <= {run_date!r}
and m.Activity_Group like '%Modifications%'
and not exists (
    select 1
    from `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) ia
    inner join `{env_prefix}-reportinputfiles`.repoint.core_qcabactivitiesmapping m on ia.InteractionTypeKey = m.Activity_Code
    where ia.InteractionDate > last_day({som_date!r})
    and ia.InteractionDate <= {run_date!r}
    and ia.ExposureKey = ia_o.ExposureKey
    and ia.InteractionDate = ia_o.InteractionDate
    and ia.InteractionTypeKey = ia_o.InteractionTypeKey
)
'''
)
interaction.createOrReplaceTempView("interaction")

##### dialer

In [0]:
dialer = spark.sql(
f'''
select distinct
        de.ExposureKey
    ,   cast('' as string) as InteractionTypeKey 
    ,   cast(DateHour as date ) as InteractionDate
    ,   DateHour as CreatedDate    
    ,   1.15 Priority
    ,   1.15 Priority_Aux 
from `{env_prefix}-cdh`.primary.dialer d
join `{env_prefix}-cdh`.primary.exposure de on cast(d.casecode as string) = cast(de.casecode as string) and d.BusinessDate = de.BusinessDate and de.sourcesystem = 'qca'
where de.PortfolioName = 'Core Retail'
and cast(d.DateHour as date ) > last_day({som_date!r})
and cast(d.DateHour as date ) <= {run_date!r}
'''
)
dialer.createOrReplaceTempView("dialer")

##### interactionwithdialer

In [0]:
interactionwithdialer = spark.sql(
f'''
select * 
    ,   CASE WHEN Priority_aux = 1.1 	then 'No Attempt'
     		WHEN Priority_aux = 1.15 	then 'Attempt'
			WHEN Priority_aux = 1.2 	then 'Attempt'
       	    WHEN Priority_aux = 2 	then 'Contact'
            WHEN Priority_aux = 3 	then 'Rest RPC Communication'
            WHEN Priority_aux = 4 	then 'Negative Communication'
            WHEN Priority_aux = 5 	then 'Positive Communication'
		END AS  StrongestAction	
    ,	RANK() OVER(PARTITION BY ExposureKey ORDER BY Priority DESC, InteractionDate DESC, CreatedDate desc, InteractionTypeKey DESC) Rank 
from (
select
        ExposureKey
    ,   InteractionTypeKey
    ,   InteractionDate
    ,   Priority
    ,   Priority_Aux
    ,   CreatedDate
from interaction

union

select         
        ExposureKey
    ,   InteractionTypeKey
    ,   InteractionDate
    ,   Priority
    ,   Priority_Aux
    ,   CreatedDate
from dialer) interactionwithdialer

'''
)
interactionwithdialer.createOrReplaceTempView("interactionwithdialer")

##### strongestaction

In [0]:
strongestaction = spark.sql(
f'''
select * 
from interactionwithdialer
where rank = 1

'''
)
strongestaction.createOrReplaceTempView("strongestaction")

##### unionfile

In [0]:
last_day_som_date = spark.sql(f'''SELECT cast(last_day({som_date!r}) as string)''').collect()[0][0]
last_day_soq_date = spark.sql(f'''SELECT cast(last_day({soq_date!r}) as string)''').collect()[0][0]

unionfile = spark.sql(
f'''
SELECT 
    uf.SnapshotDate
,	uf.ExposureNumber
,	uf.EBAStatus
,	uf.ProbationStartDate
,	uf.BucketCategoryWithMateriality
,	uf.FPLUnderProbationFlag
,   uf.ForbearanceMeasuresCount
,   uf.ingestionTimeStamp
,   case when uf.LTV = '' then null else cast(uf.LTV as decimal(19,4)) end LTV
FROM `{env_prefix}-cdh`.externalfiles.extunionfile uf
INNER JOIN (SELECT SnapshotDate, max(IngestionTimestamp) AS IngestionTimestamp
            FROM `{env_prefix}-cdh`.externalfiles.extunionfile
            WHERE SnapshotDate in (last_day({som_date!r}),last_day({soq_date!r}))
            GROUP BY SnapshotDate) uf0 
            ON uf.IngestionTimestamp = uf0.IngestionTimestamp and uf.SnapshotDate = uf0.SnapshotDate
WHERE uf.SnapshotDate in ({last_day_som_date!r},{last_day_soq_date!r})
''')
unionfile.createOrReplaceTempView("unionfile")

# Check if the dataset is empty and print a message accordingly
if unionfile.count() == 0:
    print(f"The required external file extunionfile dataset for dates {last_day_som_date!r}, {last_day_soq_date!r} is empty")

##### modifications

In [0]:
modifications = spark.sql(
f'''
SELECT * 
FROM (
SELECT 
	m.BusinessDate
,	m.OldExposureKey 
,	m.ModificationDate
,	m.Category
,	m.Categoryfinal
,	RANK() OVER (PARTITION BY OldExposureKey ORDER BY ModificationDate ASC) as rank
FROM `{env_prefix}-cdh-delivery`.repoint.core_modifications m
WHERE m.BusinessDate = {run_date!r}) a
WHERE rank = 1 

'''
)
modifications.createOrReplaceTempView("modifications")

##### npeplanningdata

In [0]:
npeplanningdata = spark.sql(
f'''
SELECT * 
FROM (
SELECT 
	m.BusinessDate
,	m.ExposureKey
,	m.Status
,	m.ImplementationDate
,	m.DateEvaluation
,   m.ModificationTypeFinal
,	rank() over(partition by m.ExposureKey, m.BusinessDate order by m.Status asc) as rank
FROM `{env_prefix}-cdh-delivery`.repoint.core_npeplanningdata m
WHERE m.BusinessDate in( {run_date!r}, {som_date!r})) m
WHERE m.rank = 1

'''
)
npeplanningdata.createOrReplaceTempView("npeplanningdata")

##### modifications_perimeter_v1

In [0]:
modifications_perimeter_v1 = spark.sql(
f'''
SELECT DISTINCT
				m.ExposureKey											 
			,	m.AssignmentChannel								
			,	m.AssignmentTeam					
			,	m.AssignmentSegmentDetailed					   
			,	m.CourseOfAction								
			,	m.AssignmentAgent
			, 	m.Anchor										
			,	m.AgreementKey																
			,	m.CDI																		
			,	m.PartyTaxRegistrationNumber																
			,	m.AssetSegment													    		
			,	m.SoMSourceBucket				  	
			,	m.EBAStatus			                
			,	m.RiskBucket	
 			,  replace(SUBSTRING(de_som.ExposureStatus, CHARINDEX('-',de_som.ExposureStatus), LEN(de_som.ExposureStatus)),'-','') AS SoMExposureStatus
 			,  m.SoMBalanceAmountEUR AS SoMBalance
			,	CASE WHEN de_mtd.Bucket = 99 OR de_mtd.ExposureStatus = 'Writen Off' THEN 0 ELSE de_mtd.Bucket END AS MtDBucket				
			,	CASE WHEN Mods.ModificationDate	>  last_day({som_date!r}) THEN Mods.Category	     ELSE NULL END		AS ModificationCategory
			,	CASE WHEN Mods.ModificationDate	>  last_day({som_date!r}) THEN Mods.Categoryfinal	 ELSE NULL END		AS ModificationProduct  
			,	CASE WHEN Mods.ModificationDate	>  last_day({som_date!r}) THEN Mods.ModificationDate ELSE NULL END		AS ImplementationDate 
			,	IFNULL(uf_soq.LTV,0) AS SoQLTV
			,   uf_som.ForbearanceMeasuresCount   
			,	de_mtd.LPS
			,	CASE WHEN de_mtd.LPS IN (
                                                           'ΕΙΣΠΡΑΞΗ ΕΚΠΛΕΙΣΤΗΡΙΑΣΜΑΤΟΣ' 
                                                          ,'ΤΕΛΕΣΙΔΙΚΙΑ ΠΙΝΑΚΑ ΚΑΤΑΞΗΣ'
                                                          ,'ΕΚΔΟΣΗ ΠΙΝΑΚΑ ΚΑΤΑΤΑΞΗΣ ΗΜΩΝ'
                                                          ,'ΔΙΕΝΕΡΓΕΙΑ ΠΛΕΙΣΤΗΡΙΑΣΜΟΥ ΗΜΩΝ'
                                                          ,'ΠΡΟΓΡΑΜΜΑ ΠΛΕΙΣΤΗΡΙΑΣΜΟΥ ΗΜΩΝ'
                                                          ,'ΕΠΙΒΟΛΗ ΚΑΤΑΣΧΕΣΗΣ'
                                                          ,'ΕΓΚΑΤΑΛΕΙΨΗ ΜΗ ΕΠΙΣΠΕΥΣH ΕΚΤΕΛΕΣΕΩΣ'
                                                          ,'ΑΝΑΚΟΠΗ ΠΛΕΙΣΤΗΡΙΑΣΜΟΥ ΗΜΩΝ'
                                                          ,'ΜΑΤΑΙΩΣΗ ΠΛΕΙΣΤΗΡΙΑΣΜΟΥ ΗΜΩΝ'
                                                          ,'ΤΕΛΕΣΙΔΙΚΙΑ ΠΙΝΑΚΑ ΚΑΤΑΤΑΞΗΣ ΤΡΙΤΟΥ'
                                                          ,'ΕΚΔΟΣΗ ΠΙΝΑΚΑ ΚΑΤΑΤΑΞΗΣ ΤΡΙΤΟΥ'
                                                          ,'ΑΝΑΛΗΨΗ ΕΠΙΣΠΕΥΣΗΣ ΠΛΕΙΣΤΗΡΙΑΣΜΟΥ'
                                                          ,'ΔΙΕΝΕΡΓΕΙΑ ΠΛΕΙΣΤΗΡΙΑΣΜΟΥ ΤΡΙΤΟΥ'
                                                          ,'ΕΚΔΟΣΗ ΠΡΟΓΡΑΜΜΑΤΟΣ ΤΡΙΤΟΥ'
                                                          ,'ΑΝΑΚΟΠΗ ΠΛΕΙΣΤΗΡΙΑΣΜΟΥ ΤΡΙΤΟΥ'
                                                          ,'ΜΑΤΑΙΩΣΗ ΠΛΕΙΣΤΗΡΙΑΣΜΟΥ ΤΡΙΤΟΥ'
                                                         ) THEN '3'
                      WHEN de_mtd.LPS IN  (
                                                           'ΕΓΓΡΑΦΗ ΠΡΟΣΗΜΕΙΩΣΗΣ'
                                                          ,'ΕΠΙΒΟΛΗ ΚΑΤΑΣΧΕΣΗΣ ΚΙΝ.ΑΠΑΙΤΗΣΕΩΝ'
                                                          ,'ΜΗ ΕΓΓΡΑΦΗ ΒΑΡΟΥΣ'
                                                          ,'ΜΗ ΕΠΙΒΟΛΗ ΚΑΤΑΣΧΕΣΗΣ ΚΙΝ.ΑΠΑΙΤΗΣΕΩΝ'
                                                          ,'ΠΡΟΣ ΕΓΓΡΑΦΗ ΒΑΡΟΥΣ'
                                                          ,'ΠΡΟΣ ΚΑΤΑΣΧΕΣΗ ΚΙΝ.ΑΠΑΙΤΗΣΕΩΝ'
                                                          ,'ΕΚΠΟΙΗΣΗ ΟΧΗΜΑΤΟΣ'
                                                          ,'ΑΦΑΙΡΕΣΗ ΟΧΗΜΑΤΟΣ'
                                                          ,'ΕΚΔΙΚΑΣΗ ΑΓΩΓΗΣ ΑΠΟΒΟΛΗΣ ΑΠΟ ΤΗ ΝΟΜΗ'
                                                          ,'ΚΑΤΑΘΕΣΗ ΑΓΩΓΗΣ ΑΠΟΒΟΛΗΣ ΑΠΟ ΤΗ ΝΟΜΗ'
                                                          ,'Β΄ ΕΠΙΔΟΣΗ Δ/Π'
                                                          ,'ΑΔΥΝΑΤΗ Β΄ ΕΠΙΔΟΣΗ Δ/Π'
                                                          ,'Α΄ ΕΠΙΔΟΣΗ Δ/Π'
                                                          ,'ΑΔΥΝΑΤΗ Α΄ ΕΠΙΔΟΣΗ Δ/Π'
                                                          ,'ΕΚΔΟΣΗ Δ/Π'
                                                          ,'ΠΡΟΣ ΣΥΝΤΑΞΗ Δ/Π'
                                                       ) THEN '2'
                      WHEN de_mtd.LPS IN (
                                                          'ΕΠΙΔΟΣΗ ΕΞΩΔΙΚΟΥ ΚΑΤΑΓΓΕΛΙΑΣ'
                                                         ,'ΜΗ ΕΠΙΔΟΣΗ ΕΞΩΔΙΚΟΥ ΚΑΤΑΓΓΕΛΙΑΣ'
                                                         ,'ΠΡΟΣ ΕΠΙΔΟΣΗ ΕΞΩΔΙΚΟΥ ΚΑΤΑΓΓΕΛΙΑΣ'
                                                         ,'ΘΕΤΙΚΗ ΕΡΕΥΝΑ ΑΚΙΝΗΤΗΣ ΠΕΡΙΟΥΣΙΑΣ'
                                                         ,'ΑΡΝΗΤΙΚΗ ΕΡΕΥΝΑ ΑΚΙΝΗΤΗΣ ΠΕΡΙΟΥΣΙΑΣ'
                                                         ,'ΕΡΕΥΝΑ ΑΚΙΝΗΤΗΣ ΠΕΡΙΟΥΣΙΑΣ'
                                                      ) THEN '1' END AS LegalphaseGroupNo
	        ,	IFNULL(attr_restr.AttributionChannel, m.AssignmentChannel)  AS FinalAssignmentChannel
	        ,	IFNULL(attr_restr.AttributionTeam, m.AssignmentTeam)  AS FinalAssignmentTeam
	        ,	IFNULL(attr_restr.AttributionCOA, m.CourseOfAction) AS FinalCourseOfAction
	        ,	IFNULL(attr_restr.AttributionSegmentDetailed, m.AssignmentSegmentDetailed) AS FinalAssignmentSegmentDetailed
			  ,	attr_pay.AttributionChannel	AS PaymentAttributionChannel
			  ,	attr_pay.AttributionTeam AS PaymentAttributionTeam
			  ,	attr_pay.PaymentAmount AS PaymentAttributionAmount
FROM AssignmentDataMapping m
INNER JOIN dexposure de_som ON de_som.ExposureKey = m.ExposureKey and de_som.BusinessDate = {som_date!r}
LEFT JOIN dexposure de_mtd ON de_mtd.ExposureKey = m.ExposureKey and de_mtd.BusinessDate = {run_date!r}
LEFT JOIN unionfile uf_som ON uf_som.ExposureNumber = m.ExposureKey and uf_som.SnapshotDate = {som_date!r}
LEFT JOIN unionfile uf_soq ON uf_soq.ExposureNumber = m.ExposureKey and uf_soq.SnapshotDate = {soq_date!r}
LEFT JOIN attributionchannel_restructure attr_restr ON attr_restr.ExposureKey = m.ExposureKey 
LEFT JOIN attributionchannel_payment attr_pay ON attr_pay.ExposureKey = m.ExposureKey
LEFT JOIN modifications Mods ON Mods.OldExposureKey = m.ExposureKey
WHERE de_som.Bucket > 0
AND de_som.ExposureStatus <> 'Writen Off'
'''
)
modifications_perimeter_v1.createOrReplaceTempView("modifications_perimeter_v1")

##### modifications_perimeter_v2

In [0]:
modifications_perimeter_v2 = spark.sql(
f'''
SELECT
        m.*
    ,	CASE WHEN m.EBAStatus IN ('FNPL', 'NPL')			THEN 'NPE'
			 WHEN m.RiskBucket = 3						THEN 'Current Month Inflows'
															ELSE 'PE'                                        
		END AS EBAIndicator
    ,   CASE WHEN m.SoMExposureStatus = 'Define Delay' THEN 'denounced'
             WHEN m.SoMSourceBucket < 3                  THEN '1-2'
             WHEN m.SoMSourceBucket < 7                  THEN '3-6'
             WHEN m.SoMSourceBucket < 13                 THEN '7-12' 
			                                       ELSE 'op13'                                   
        END AS DelinquencyOperational
	,   CASE WHEN m.Anchor = 'Anchor' THEN 'anchorProduct' ELSE 'non modified/non anchor' END AS PreviousProduct
	,   CASE WHEN m.ForbearanceMeasuresCount > 0 THEN 'modified' ELSE 'non modified' END AS ModifiedInput
	,   CASE WHEN m.SoQLTV = 0 AND m.AssetSegment != 'ML'   THEN 'Unsecured' 
			 WHEN m.SoQLTV >= 1.2 THEN 'LTV>=120%'
             WHEN m.SoQLTV <  1.2 THEN 'LTV<120%'
                                  ELSE 'LTV>=120%'                              
        END AS LTVFlag
	,   CASE WHEN m.SoMExposureStatus = 'Define Delay' AND (m.LegalphaseGroupNo = '2' OR m.LegalphaseGroupNo = '3') THEN 'after po'  ELSE 'before po' END AS Poinput
	,   CASE WHEN m.SoMExposureStatus = 'Define Delay' THEN 'denounced'
			 WHEN m.RiskBucket <  4             THEN '0-3'
			 WHEN m.RiskBucket >= 4             THEN '4+' 
        END AS DelinquencyRisk		
FROM modifications_perimeter_v1 m
'''
)
modifications_perimeter_v2.createOrReplaceTempView("modifications_perimeter_v2")

##### modifications_perimeter_v3

In [0]:
core_collectiontargets = spark.sql(
f'''
SELECT * 
FROM (
SELECT TargetType
      ,RiskBucket
      ,OpBucket
      ,EBA
      ,IF(CollectionTarget IS NOT NULL,CONCAT('0',REPLACE(CollectionTarget,',','.')),0.000000) CollectionTarget
      ,IF(Extension IS NOT NULL,CONCAT('0',REPLACE(Extension,',','.')),0.000000) Extension
      ,IF(Restructuring IS NOT NULL,CONCAT('0',REPLACE(Restructuring,',','.')),0.000000) Restructuring
      ,IF(Unhandled IS NOT NULL,CONCAT('0',REPLACE(Unhandled,',','.')),0.000000) Unhandled
      ,IF(DTTotal IS NOT NULL,CONCAT('0',REPLACE(DTTotal,',','.')),0.000000) DTTotal
      ,IF(DTCollection IS NOT NULL,CONCAT('0',REPLACE(DTCollection,',','.')),0.000000) DTCollection
      ,ROW_NUMBER() OVER(PARTITION BY BusinessDate, TargetType, EBA, RiskBucket, OpBucket ORDER BY IngestionTimeStamp) AS Rank
      
FROM `{env_prefix}-reportinputfiles`.repoint.core_collectiontargets 
WHERE BusinessDate = (SELECT MAX(BusinessDate) FROM `{env_prefix}-reportinputfiles`.repoint.core_collectiontargets WHERE BusinessDate <= {run_date!r})) A
WHERE Rank = 1
'''
)
core_collectiontargets.createOrReplaceTempView("core_collectiontargets")

In [0]:
ExternalModifications = spark.sql(
f'''
select distinct em.Portfolio, em.CIF, em.AFM, em.EBAStatus, em.AccountNumber, cast(em.ApplicationFlag as int) ApplicationFlag, 
em.FinalSaved, em.SavedReason, em.ImplementationDate, em.AttributionChannel, em.AttributionTeam, em.AttributionAgent, case when lower(em.SavedDate) in('null','') then null else cast(em.SavedDate as date) end as SavedDate, cast(em.SomBalance as decimal(19,2)) as SomBalance, em.ExtraDPOFlag   
from `{env_prefix}-reportinputfiles`.repoint.core_externalfileattributions em
WHERE em.ReferenceDateEOM = LAST_DAY({run_date!r})
''')   
ExternalModifications.createOrReplaceTempView("ExternalModifications")

In [0]:
modifications_perimeter_v3 = spark.sql(
f'''
SELECT
            m.ExposureKey
        ,   m.AssignmentChannel
        ,   m.AssignmentTeam
        ,   m.AssignmentSegmentDetailed
        ,   m.CourseOfAction
        ,   m.AssignmentAgent
        ,   m.Anchor
        ,   m.AgreementKey
        ,   ifnull(em.CIF, m.CDI) AS CDI
        ,   ifnull(em.AFM, m.PartyTaxRegistrationNumber ) AS PartyTaxRegistrationNumber
        ,   m.AssetSegment
        ,   m.SoMSourceBucket
        ,   ifnull(em.EBAStatus, m.EBAStatus) AS EBAStatusUpdated
        ,   m.RiskBucket
        ,   m.SoMExposureStatus
        ,   ifnull(em.SoMBalance, m.SoMBalance) AS SoMBalanceUpdated
        ,   m.MtDBucket
        ,   m.ModificationCategory
        ,   case when em.ExtraDPOFlag = 'Yes' then 'DPO' else m.ModificationProduct end AS ModificationProductUpdated
        ,   ifnull(em.ImplementationDate, m.ImplementationDate) AS ImplementationDate
        ,   m.SoQLTV
        ,   m.ForbearanceMeasuresCount
        ,   m.LPS
        ,   m.LegalphaseGroupNo
        ,   ifnull(em.AttributionChannel, m.FinalAssignmentChannel) AS FinalAssignmentChannelUpdated
        ,   ifnull(em.AttributionTeam, m.FinalAssignmentTeam) AS FinalAssignmentTeam
        ,   m.FinalCourseOfAction
        ,   m.FinalAssignmentSegmentDetailed
        ,   case when em.SavedReason = 'Payments' then em.AttributionChannel else m.PaymentAttributionChannel end AS PaymentAttributionChannel
        ,   case when em.SavedReason = 'Payments' then em.AttributionTeam else m.PaymentAttributionTeam end AS PaymentAttributionTeam
        ,   m.PaymentAttributionAmount
        ,   m.EBAIndicator
        ,   m.DelinquencyOperational
        ,   m.PreviousProduct
        ,   m.ModifiedInput
        ,   m.LTVFlag
        ,   m.Poinput
        ,   m.DelinquencyRisk
        ,   CASE WHEN m.PreviousProduct = 'anchorProduct' AND m.DelinquencyOperational = 'denounced' THEN CONCAT('Anchor_NPE_denounced_', m.LTVFlag,'_',m.Poinput)
                     WHEN m.PreviousProduct = 'anchorProduct'     THEN 'Anchor_NPE_4+'
                     WHEN m.DelinquencyOperational = 'denounced'  THEN CONCAT('NPE_',m.DelinquencyOperational,'_',m.LTVFlag,'_',m.Poinput)
                     WHEN m.DelinquencyOperational = 'op13'       THEN CONCAT('NPE_4+','_',m.DelinquencyOperational,'_',m.LTVFlag,'_',m.ModifiedInput)
                     ELSE CONCAT('NPE_4+_3-12','_',m.LTVFlag,'_',m.ModifiedInput,'_',m.DelinquencyOperational)
                END AS Segment
        ,   case when m.ImplementationDate is not null or last_day(D.DateEvaluation) = last_day({run_date!r}) then 'Positive Communication' else SA.StrongestAction end AS ContactMtD
        ,   ifnull(em.FinalSaved, S.FinalSaved) AS FinalSavedFlag
		,	ifnull(em.SavedDate, S.FinalSavedDate) AS FinalSavedDate  
		,   ifnull(em.SavedReason, S.SavedReason) AS FinalSavedReason
        ,   MI.TargetDate
		,   CASE WHEN S.FinalSaved = 'Not Saved' AND em.FinalSaved is null AND m.EBAIndicator = 'Current Month Inflows' AND {run_date!r} >= MI.TargetDate THEN '1' ELSE '0' END AS NPEFinalizedInflow 
        ,   CASE WHEN A.Status = 'Ενεργή' AND A.ImplementationDate IS NULL THEN A.Status ELSE NULL END AS SoMStatus   
    	,   CASE WHEN D.DateEvaluation <= {som_date!r} THEN NULL 
                 WHEN em.ApplicationFlag = 1 THEN 'Ολοκληρωμένη'
                 ELSE D.Status END AS MtDProductionStatus
		,   CASE WHEN D.DateEvaluation <= {som_date!r} THEN NULL ELSE D.DateEvaluation END AS MtDProductionDate
		,   CASE WHEN D.Status = 'Ενεργή' AND D.ImplementationDate IS NULL  THEN D.Status ELSE NULL END AS CurrentStatus			  
		,	D.ModificationTypeFinal AS ApplicationProduct	
        ,   CASE WHEN FinalAssignmentChannelUpdated = 'IRT' AND (m.FinalAssignmentSegmentDetailed = 'PJ_Extension' OR m.FinalAssignmentSegmentDetailed = 'PJ_Extension_Stepup' OR m.FinalAssignmentSegmentDetailed = 'LowInstallment')   THEN 'IRT - Inst. Extension'
                 WHEN FinalAssignmentChannelUpdated = 'IRT' AND (m.FinalAssignmentSegmentDetailed = 'DPO' OR m.FinalAssignmentSegmentDetailed ='DPO_PE')  THEN 'IRT - DPO'
                 WHEN (m.FinalCourseOfAction = 'Double Treatment' OR m.FinalCourseOfAction = 'Restructuring' OR m.FinalAssignmentSegmentDetailed = 'N3869_ΔΤ' OR  MtDProductionStatus='Ενεργή' OR ModificationProductUpdated is not null  ) AND (FinalAssignmentChannelUpdated like 'HUB RMs%' or FinalAssignmentChannelUpdated like 'Hub RMs%') THEN 'Hub RMs'
                 WHEN (m.FinalCourseOfAction = 'Double Treatment' OR m.FinalCourseOfAction = 'Restructuring' OR  MtDProductionStatus='Ενεργή' OR ModificationProductUpdated is not null )  AND FinalAssignmentChannelUpdated ='IRT' THEN 'IRT'
                 WHEN (m.FinalCourseOfAction = 'Double Treatment' OR m.FinalCourseOfAction = 'Restructuring' OR  MtDProductionStatus='Ενεργή' OR ModificationProductUpdated is not null ) AND FinalAssignmentChannelUpdated = 'Call Center' THEN 'Call Center'
                 WHEN (m.FinalCourseOfAction = 'Double Treatment' OR m.FinalCourseOfAction = 'Restructuring' OR  MtDProductionStatus='Ενεργή' OR ModificationProductUpdated is not null ) AND FinalAssignmentChannelUpdated = 'Legal Agencies'  THEN 'Legal Agencies'
                 WHEN (FinalAssignmentChannelUpdated = 'Legal Agencies' OR  MtDProductionStatus='Ενεργή' OR ModificationProductUpdated is not null )  THEN 'Legal Agencies' 
                 ELSE 'Rest' END AS NewAssignmentChannel
		,   CASE WHEN EBAStatusUpdated NOT LIKE '%NPL' AND NewAssignmentChannel =  'IRT - Inst. Extension' THEN 0 
			     WHEN EBAStatusUpdated NOT LIKE '%NPL' AND NewAssignmentChannel <> 'IRT - Inst. Extension' THEN SoMBalanceUpdated * (T.DTTotal-T.DTCollection) * 1.2 
			END ApplicationTarget
		,   CASE WHEN EBAStatusUpdated NOT LIKE '%NPL' AND NewAssignmentChannel =  'IRT - Inst. Extension' THEN SoMBalanceUpdated * T.Restructuring 
			     WHEN EBAStatusUpdated NOT LIKE '%NPL' AND NewAssignmentChannel <> 'IRT - Inst. Extension' THEN SoMBalanceUpdated * (T.DTTotal-T.DTCollection)
			END ImplementationTarget
		,   CASE WHEN EBAStatusUpdated NOT LIKE '%NPL' AND FinalAssignmentSegmentDetailed IN ('PJ_Extension','PJ_Extension_Stepup','LowInstallment')		
               THEN SoMBalanceUpdated * T.Extension 
			     WHEN EBAStatusUpdated NOT LIKE '%NPL' AND ifnull(FinalAssignmentSegmentDetailed,'') NOT IN ('PJ_Extension','PJ_Extension_Stepup','LowInstallment')	THEN SoMBalanceUpdated * T.DTTotal
			END SavedTarget 
FROM modifications_perimeter_v2 m
LEFT JOIN SavedInfo S ON S.ExposureKey = m.ExposureKey	
LEFT JOIN maxinflows MI ON MI.ExposureNumber = m.ExposureKey 	
LEFT JOIN npeplanningdata A ON A.ExposureKey = m.ExposureKey AND A.BusinessDate = {som_date!r}
LEFT JOIN npeplanningdata D ON D.ExposureKey = m.ExposureKey AND D.BusinessDate = {run_date!r} 
LEFT JOIN strongestaction SA ON SA.ExposureKey = m.ExposureKey
LEFT JOIN core_collectiontargets T ON m.EBAStatus = T.EBA AND IF(m.RiskBucket>=4,'4+',CAST(m.RiskBucket AS STRING)) = T.RiskBucket AND IF(m.SoMSourceBucket>=4,'4+',CAST(m.SoMSourceBucket AS STRING)) = T.OpBucket 
LEFT JOIN ExternalModifications em on em.AccountNumber = m.ExposureKey

UNION ALL

SELECT
            em.AccountNumber AS ExposureKey
        ,   NULL AS AssignmentChannel
        ,   NULL AS AssignmentTeam
        ,   NULL AS AssignmentSegmentDetailed
        ,   NULL AS CourseOfAction
        ,   NULL AS AssignmentAgent
        ,   NULL AS Anchor
        ,   NULL AS AgreementKey
        ,   em.CIF AS CDI
        ,   em.AFM AS PartyTaxRegistrationNumber
        ,   NULL AS AssetSegment
        ,   NULL AS SoMSourceBucket
        ,   em.EBAStatus AS EBAStatusUpdated
        ,   NULL AS RiskBucket
        ,   NULL AS SoMExposureStatus
        ,   em.SoMBalance AS SoMBalanceUpdated
        ,   NULL AS MtDBucket
        ,   NULL AS ModificationCategory
        ,   case when em.ExtraDPOFlag = 'Yes' then 'DPO' end AS ModificationProductUpdated
        ,   em.ImplementationDate AS ImplementationDate
        ,   NULL AS SoQLTV
        ,   NULL AS ForbearanceMeasuresCount
        ,   NULL AS LPS
        ,   NULL AS LegalphaseGroupNo
        ,   em.AttributionChannel AS FinalAssignmentChannelUpdated
        ,   em.AttributionTeam AS FinalAssignmentTeam
        ,   NULL AS FinalCourseOfAction
        ,   NULL AS FinalAssignmentSegmentDetailed
        ,   case when em.SavedReason = 'Payments' then em.AttributionChannel end AS PaymentAttributionChannel
        ,   case when em.SavedReason = 'Payments' then em.AttributionTeam end AS PaymentAttributionTeam
        ,   NULL AS PaymentAttributionAmount
        ,   NULL AS EBAIndicator
        ,   NULL AS DelinquencyOperational
        ,   NULL AS PreviousProduct
        ,   NULL AS ModifiedInput
        ,   NULL AS LTVFlag
        ,   NULL AS Poinput
        ,   NULL AS DelinquencyRisk
        ,   NULL AS Segment
        ,   NULL AS ContactMtD
        ,   em.FinalSaved AS FinalSavedFlag
		,	em.SavedDate AS FinalSavedDate  
		,   em.SavedReason AS FinalSavedReason
        ,   NULL AS TargetDate
		,   NULL AS NPEFinalizedInflow 
        ,   NULL AS SoMStatus   
    	,   CASE WHEN em.ApplicationFlag = 1 THEN 'Ολοκληρωμένη' END AS MtDProductionStatus
		,   NULL AS MtDProductionDate
		,   NULL AS CurrentStatus			  
		,	NULL AS ApplicationProduct	
        ,   FinalAssignmentChannelUpdated AS NewAssignmentChannel
		,   NULL AS ApplicationTarget
		,   NULL AS ImplementationTarget
		,   NULL AS SavedTarget 
FROM ExternalModifications em
LEFT JOIN AssignmentDataMapping a on em.AccountNumber = a.ExposureKey
WHERE a.ExposureKey is null

'''
)
modifications_perimeter_v3.createOrReplaceTempView("modifications_perimeter_v3")

In [0]:
final_df = spark.sql(
f'''
SELECT CAST({run_date!r} AS DATE) AS BusinessDate,
CAST(ExposureKey AS STRING) AS ExposureKey,
CAST(Segment AS STRING) AS Segment,
CAST(LTVFlag AS STRING) AS LTVFlag,
CAST(MtDBucket AS INT) AS MtDBucket,
CAST(RiskBucket AS INT) AS ReportedBucket,
CAST(AgreementKey AS STRING) AS AgreementKey,
CAST(CDI AS STRING) AS CDI,
CAST(PartyTaxRegistrationNumber AS STRING) AS PartyTaxRegistrationNumber,
CAST(EBAIndicator AS STRING) AS EBAIndicator,
CAST(EBAStatusUpdated AS STRING) AS EBAStatus,
CAST(ContactMtD AS STRING) AS ContactMtD,
CAST(AssetSegment AS STRING) AS AssetSegment,
CAST(SoMSourceBucket AS INT) AS SoMBucket,
CAST(SoMExposureStatus AS STRING) AS SoMDenouncedStatus,
CAST(SoMBalanceUpdated AS DECIMAL(19,2)) AS SoMBalance,
CAST(ModificationCategory AS STRING) AS ModificationCategory,
CAST(ModificationProductUpdated AS STRING) AS ModificationProduct,
CAST(ImplementationDate AS DATE) AS ImplementationDate,
CAST(NewAssignmentChannel AS STRING) AS NewAssignmentChannel,
CAST(AssignmentChannel AS STRING) AS AssignmentChannel,
CAST(AssignmentTeam AS STRING) AS AssignmentTeam,
CAST(AssignmentAgent AS STRING) AS AssignmentAgent,
CAST(CourseOfAction AS STRING) AS CourseOfAction,
CAST(AssignmentSegmentDetailed AS STRING) AS AssignmentSegmentDetailed,
CAST(SoMStatus AS STRING) AS SoMStatus,
CAST(MtDProductionStatus AS STRING) AS MtDProductionStatus,
CAST(MtDProductionDate AS DATE) AS MtDProductionDate,
CAST(CurrentStatus AS STRING) AS CurrentStatus,
CAST(ApplicationProduct AS STRING) AS ApplicationProduct,
CAST(FinalSavedFlag AS STRING) AS FinalSavedFlag,
CAST(FinalSavedReason AS STRING) AS FinalSavedReason,
CAST(FinalSavedDate AS DATE) AS FinalSavedDate,
CAST(TargetDate AS DATE) AS TargetDate,
CAST(NPEFinalizedInflow AS INT) AS NPEFinalizedInflow,
CAST(FinalAssignmentChannelUpdated AS STRING) AS FinalAssignmentChannel,
CAST(FinalAssignmentTeam AS STRING) AS FinalAssignmentTeam,
CAST(PaymentAttributionChannel AS STRING) AS PaymentAttributionChannel,
CAST(PaymentAttributionTeam AS STRING) AS PaymentAttributionTeam,
CAST(PaymentAttributionAmount AS DECIMAL(19,2)) AS PaymentAttributionAmount,
CAST(ApplicationTarget AS DECIMAL(19,2)) AS ApplicationTarget,
CAST(ImplementationTarget AS DECIMAL(19,2)) AS ImplementationTarget,
CAST(SavedTarget AS DECIMAL(19,2)) AS SavedTarget,
CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp
from modifications_perimeter_v3
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_PMModifications: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
PMModifications = spark.sql(
f'''
select * from {delivery_table_path}
--where BusinessDate = {run_date!r} na mpei sthn paragwgh
'''
)
PMModifications.createOrReplaceTempView("PMModifications")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/PMModifications/"
file_name = "PMModifications_" +run_date+ ".csv"
PMModifications.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)